To calculate the tidal flow for Imperial Beach, I will use the `noaa_coops` library (if available, otherwise standard requests to NOAA CO-OPS API) to fetch hourly tide predictions. Then, I will process this data to determine the tidal state (flood, ebb, slack high, slack low) based on the change in height.

First, I need to install `noaa_coops` to easily fetch the data.



In [7]:
!pip install noaa_coops
!pip list

Package                           Version
--------------------------------- ------------
aiobotocore                       2.7.0
aiofiles                          24.1.0
aiohttp                           3.9.3
aioitertools                      0.7.1
aiosignal                         1.2.0
alabaster                         0.7.12
altair                            5.0.1
anaconda-anon-usage               0.4.3
anaconda-catalogs                 0.2.0
anaconda-client                   1.12.3
anaconda-cloud-auth               0.1.4
anaconda-navigator                2.5.4
anaconda-project                  0.11.1
annotated-types                   0.7.0
anthropic                         0.53.0
anyio                             4.2.0
appdirs                           1.4.4
applaunchservices                 0.3.0
appnope                           0.1.2
appscript                         1.1.2
archspec                          0.2.3
argon2-cffi                       21.3.0
argon2-cffi-bindings     

In [16]:
import pandas as pd
from noaa_coops import Station
import datetime

# Imperial Beach, CA Station ID: 9410120
# If 9410120 is not available for predictions, we might use a nearby reference station like San Diego (9410170)
# and apply corrections, but let's try Imperial Beach first or the nearest active station with predictions.
# San Diego, Quaker St. is 9410170. Imperial Beach is a subordinate station usually.
# Let's use San Diego (9410170) as it is a primary reference station often used for the region
# if the specific IB station doesn't provide harmonic predictions via this API easily.
# However, usually for "Imperial Beach", San Diego (9410170) is the reference.
# Let's try to get data for San Diego (9410170) which is the standard reference for the bay/coast nearby.

#STATION_ID = "9410120"  # Imperial Beach, CA
STATION_ID = "9410170"  # San Diego, San Diego Bay

start_date = datetime.date.today().strftime('%Y%m%d')
end_date = (datetime.date.today() + datetime.timedelta(days=7)).strftime('%Y%m%d')

station = Station(id=STATION_ID)

# Fetch predictions (hourly is standard for 'predictions' endpoint usually with interval='h')
df_tides = station.get_data(
    begin_date=start_date,
    end_date=end_date,
    product="predictions",
    datum="MLLW",
    interval="h",
    units="english",
    time_zone="gmt"
)

# Reset index to make time a column
df_tides = df_tides.reset_index().rename(columns={'t': 'time', 'v': 'tide height'})

# Calculate the difference between consecutive hours to determine state
df_tides['diff'] = df_tides['tide height'].diff()


def determine_state(diff):
    threshold = 0.1  # Small threshold for "slack" approximation if needed, or strictly use sign
    if pd.isna(diff):
        return None
    elif abs(diff) < 0.05:  # Very small change could be considered slack periods in hourly data
        # To be more precise about "slack high" vs "slack low", we need to look at the trend before/after
        # But based on simple derivative:
        return "slack"
    elif diff > 0:
        return "flood"
    else:
        return "ebb"


# A better way to define Slack High/Low on hourly data is looking at peaks/troughs.
# If t is higher than t-1 and t+1, it's near high tide.
# However, the request asks for state for *each hour*.
# "Flood" means rising, "Ebb" means falling.
# "Slack" usually happens exactly at the turn, which might not align with an exact hour.
# We will approximate:
# If direction changes from + to -, the point before was High Slack.
# If direction changes from - to +, the point before was Low Slack.
# For the purpose of "flow", strictly:
# - Rising tide = Flood
# - Falling tide = Ebb
# - Near zero change = Slack

# Let's refine the state logic to include High/Low slack based on peaks.
# We will identify local maxima and minima.

df_tides['state'] = df_tides['diff'].apply(lambda x: 'flood' if x > 0 else ('ebb' if x < 0 else 'slack'))

# Refine slack high/low
# We can find where the derivative changes sign
# Create a shifted column to compare current slope with next slope
df_tides['next_diff'] = df_tides['diff'].shift(-1)


def refine_state(row):
    # This logic checks if the current hour is a turning point
    # Since data is hourly, the exact slack might be between hours,
    # but we can label the hour closest to the turn or the turning interval.

    # Simple approach:
    # If height increases then decreases -> High (Slack High)
    # If height decreases then increases -> Low (Slack Low)

    current_diff = row['diff']
    next_diff = row['next_diff']

    if pd.isna(current_diff) or pd.isna(next_diff):
        return row['state']

    # Check for High Slack (transition from flood to ebb)
    if current_diff > 0 and next_diff < 0:
        return "slack high"

    # Check for Low Slack (transition from ebb to flood)
    elif current_diff < 0 and next_diff > 0:
        return "slack low"

    return row['state']


df_tides['state'] = df_tides.apply(refine_state, axis=1)

# Clean up columns
df_final = df_tides[['time', 'tide height', 'state']]

df_final


,time,tide height,state
0,2025-12-08 00:00:00,-0.431,slack
1,2025-12-08 01:00:00,-1.098,slack low
2,2025-12-08 02:00:00,-1.082,flood
3,2025-12-08 03:00:00,-0.449,flood
4,2025-12-08 04:00:00,0.600,flood
...,...,...,...
164,2025-12-14 20:00:00,0.917,slack low
165,2025-12-14 21:00:00,0.944,flood
166,2025-12-14 22:00:00,1.322,flood
167,2025-12-14 23:00:00,1.957,flood


To address your request, I will add two new cells:

1.  **Data Inventory**: I will retrieve and display the data inventory for the **Imperial Beach** station (ID: 9410120) to see what data products are available.
2.  **Historic Data**: I will fetch the historic hourly tide predictions for **2023 and 2024** using the San Diego reference station (ID: 9410170), as established in the previous step for flow calculations. I will then apply the same logic to calculate the `state` (flood, ebb, slack high, slack low) for this historic dataset.



In [11]:
# Cell 1: Data Inventory for Imperial Beach (Station ID: 9410120)
try:
    ib_station = Station(id="9410120")  # Imperial Beach
    # Retrieve data inventory
    inventory = ib_station.data_inventory
    print("Data Inventory for Imperial Beach (9410120):")
    print(inventory)
except Exception as e:
    print(f"Could not retrieve inventory for Imperial Beach: {e}")

Data Inventory for Imperial Beach (9410120):
{'Verified Monthly Mean Water Level': {'start_date': '1970-11-01 00:00', 'end_date': '1979-12-31 23:54'}}


In [17]:
# Cell 2: Historic Hourly Data for 2023 and 2024 (using Reference Station 9410170)
# We use the 'station' object defined previously (San Diego) to ensure data availability for flow calc.

# Define historic range
start_hist = "20230101"
end_hist = "20241231"

print(f"Fetching historic data from {start_hist} to {end_hist}...")

# Fetch predictions
df_hist = station.get_data(
    begin_date=start_hist,
    end_date=end_hist,
   # product="predictions",
     product="hourly_height",
    datum="MLLW",
    #interval="h",
    units="english",
    time_zone="lst_ldt"
)

# Process dataframe matching the requested structure
df_hist = df_hist.reset_index().rename(columns={'t': 'time', 'v': 'tide height'})

# Calculate differences to determine state
df_hist['diff'] = df_hist['tide height'].diff()

# Initial state assignment
df_hist['state'] = df_hist['diff'].apply(lambda x: 'flood' if x > 0 else ('ebb' if x < 0 else 'slack'))

# Prepare for refinement (looking ahead to find peaks/troughs)
df_hist['next_diff'] = df_hist['diff'].shift(-1)

# Refine state using the previously defined function 'refine_state'
df_hist['state'] = df_hist.apply(refine_state, axis=1)

# specific structure requested
df_hist_final = df_hist[['time', 'tide height', 'state']]

df_hist_final


Fetching historic data from 20230101 to 20241231...


,time,tide height,state
0,2023-01-01 00:00:00,3.050,slack
1,2023-01-01 01:00:00,3.716,flood
2,2023-01-01 02:00:00,4.674,flood
3,2023-01-01 03:00:00,5.613,flood
4,2023-01-01 04:00:00,6.095,flood
...,...,...,...
17516,2024-12-30 20:00:00,2.922,flood
17517,2024-12-30 21:00:00,3.677,flood
17518,2024-12-30 22:00:00,3.798,slack high
17519,2024-12-30 23:00:00,3.533,ebb


# note there is a currents and current_prediction dataset, but thos are not available in the region.